In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from itertools import combinations
from datetime import datetime

| Column | Description |
|--------|-------------|
| **customer_id** | Unique customer identifier |
| **first_name** | Customer first name |
| **last_name** | Customer last name |
| **gender** | Male, Female, Other |
| **age_group** | Teenagers, Adults, Senior |
| **signup_date** | Customer registration date |
| **country** | Customer country (Brazil, Canada, China, France, Germany, UK, USA) |
| **product_id** | Unique product identifier |
| **product_name** | Product name |
| **category** | Product category (Electronics, Apparel, Toys, Home & Kitchen, Books) |
| **quantity** | Number of units purchased |
| **unit_price** | Price per unit |
| **order_id** | Unique order identifier |
| **order_date** | Transaction date |
| **order_status** | Delivered, Pending, Returned, Cancelled |
| **payment_method** | Credit Card, PayPal, Cash on Delivery |
| **rating** | Customer rating (1-5) |
| **review_text** | Text review (good, average, very good, bad) |
| **review_id** | Unique review identifier |
| **review_date** | Date review was posted |

In [2]:
# ref: https://www.kaggle.com/datasets/nabihazahid/ecommerce-dataset-for-sql-analysis
data = pd.read_csv('../datasets/ecommerce_dataset_10000.csv', parse_dates=['order_date', 'signup_date'])
data

,customer_id,first_name,last_name,gender,age_group,signup_date,country,product_id,product_name,category,quantity,unit_price,order_id,order_date,order_status,payment_method,rating,review_text,review_id,review_date
0,CUST2353,Erica,Oliver,Female,Teenagers,2022-06-29,Canada,PROD108,Fitbit Versa 3,Electronics,3,229,ORD10000,2023-07-13,Pending,Credit Card,2,good,REV20000,2025-06-06
1,CUST4463,Christopher,White,Male,Adults,2023-08-24,China,PROD103,Levi's Jeans,Apparel,4,59,ORD10001,2024-08-12,Pending,PayPal,2,average,REV20001,2023-08-05
2,CUST4512,Spencer,Foster,Male,Senior,2023-07-18,Germany,PROD111,Lego Star Wars Set,Toys,2,59,ORD10002,2024-08-04,Delivered,Cash on Delivery,5,good,REV20002,2023-01-03
3,CUST5711,Jessica,Harris,Male,Teenagers,2025-08-22,France,PROD107,Dyson Vacuum,Home & Kitchen,4,399,ORD10003,2025-05-23,Delivered,Cash on Delivery,2,very good,REV20003,2023-03-14
4,CUST1296,Amy,Johnson,Female,Teenagers,2021-03-23,Brazil,PROD105,Adidas Running Shoes,Apparel,1,110,ORD10004,2023-07-02,Returned,Cash on Delivery,1,very good,REV20004,2023-10-18
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,CUST2529,Lisa,Coleman,Male,Senior,2023-01-06,UK,PROD109,Kindle Paperwhite,Books,5,129,ORD19995,2023-06-07,Delivered,PayPal,4,very good,REV29995,2024-07-30
9996,CUST4602,Stacy,Brown,Other,Teenagers,2023-08-30,Canada,PROD109,Kindle Paperwhite,Books,5,129,ORD19996,2025-08-05,Returned,PayPal,1,bad,REV29996,2024-09-13
9997,CUST1448,Sheila,Roberts,Other,Adults,2024-05-18,USA,PROD107,Dyson Vacuum,Home & Kitchen,5,399,ORD19997,2023-12-23,Returned,Cash on Delivery,3,average,REV29997,2023-09-13
9998,CUST1343,Paul,Lam,Male,Teenagers,2024-07-14,Canada,PROD107,Dyson Vacuum,Home & Kitchen,3,399,ORD19998,2022-08-30,Cancelled,Credit Card,2,very good,REV29998,2025-07-07


### Build a comprehensive customer segmentation model based on purchase behavior and lifetime value.

##### Step 1: Calculate Customer Metrics

For each customer, calculate the following:

| Metric | Formula / Method |
|--------|------------------|
| **Total Revenue** | Sum of `(quantity × unit_price)` across all orders |
| **Total Orders** | Number of unique `order_id` per customer |
| **Average Order Value (AOV)** | `Total Revenue / Total Orders` |
| **Purchase Frequency** | `(Days between first and last order) / Total Orders` |
| **Recency** | Days since last purchase (using `today's date`) |
| **Product Diversity** | Number of unique `category` purchased |
| **Return Rate** | Percentage of orders where `order_status = 'Returned'` |


##### Step 2: Create Customer Value Score (0-100)

Weighted combination of normalized metrics:

| Component | Weight | Description |
|-----------|--------|-------------|
| **Monetary** | 30% | Total revenue (normalized) |
| **Frequency** | 25% | Number of orders (normalized) |
| **Recency** | 25% | Inverse recency (normalized) |
| **Product Diversity** | 20% | Unique categories (normalized) |

$$
\text{Value Score} = 0.30 \times \text{Monetary}_\text{norm} + 0.25 \times \text{Frequency}_\text{norm} + 0.25 \times \text{Recency}_\text{norm} + 0.20 \times \text{Diversity}_\text{norm}
$$


##### Step 3: Segment Customers

| Segment | Criteria |
|---------|----------|
| **Champions** | Top 10% by value score |
| **Loyal Customers** | High frequency, medium-high value |
| **Potential Loyalists** | Recent customers with high value |
| **At Risk** | High value but low recency |
| **Hibernating** | Low recency, low frequency, low value |


##### Step 4: Analyze Each Segment

For each segment, analyze:

| Metric | Description |
|--------|-------------|
| **Average Rating** | Mean rating given by customers in the segment |
| **Most Preferred Category** | Most purchased category |
| **Favorite Payment Method** | Most used payment method |
| **Order Status Distribution** | Breakdown of order statuses |


In [3]:
data['revenue'] = data['quantity'] * data['unit_price']

customer_metrics = data.groupby(['customer_id']).agg(
    total_revenue = ('revenue', 'sum'),
    total_orders = ('order_id', 'nunique'), 
    first_order_date = ('order_date', 'min'),
    last_order_date = ('order_date', 'max'),
    days_between_first_and_last_order_date = ('order_date', lambda x: (x.max() - x.min()).days),
    days_since_last_purchase_recency = ('order_date', lambda x: (datetime.now() - x.max()).days),
    category_diversity = ('category', 'nunique'),
    return_rate_pct = ('order_status', lambda x: (x == 'Returned').mean() * 100)
).reset_index()

customer_metrics = customer_metrics.assign(
    aov=lambda customer_metrics: (customer_metrics['total_revenue'] / customer_metrics['total_orders']).round(2),
    purchase_frequency=lambda df: (customer_metrics['days_between_first_and_last_order_date'] / customer_metrics['total_orders']).round(2)
)
# customer_metrics['aov'] = (customer_metrics['total_revenue'] / customer_metrics['total_orders']).round(2)
# customer_metrics['purchase_frequency'] = (customer_metrics['days_between_first_and_last_order_date'] / customer_metrics['total_orders']).round(2)
customer_metrics

,customer_id,total_revenue,total_orders,first_order_date,last_order_date,days_between_first_and_last_order_date,days_since_last_purchase_recency,category_diversity,return_rate_pct,aov,purchase_frequency
0,CUST1000,6336,5,2022-12-29,2024-12-30,732,588,3,40.0,1267.20,146.40
1,CUST1001,690,2,2024-11-08,2025-04-08,151,489,1,0.0,345.00,75.50
2,CUST1002,999,1,2022-11-09,2022-11-09,0,1370,1,0.0,999.00,0.00
3,CUST1003,444,2,2023-06-05,2023-12-30,208,954,2,50.0,222.00,104.00
4,CUST1004,796,1,2025-01-14,2025-01-14,0,573,1,0.0,796.00,0.00
...,...,...,...,...,...,...,...,...,...,...,...
4322,CUST5993,796,1,2023-09-30,2023-09-30,0,1045,1,100.0,796.00,0.00
4323,CUST5994,2342,3,2022-09-10,2024-01-19,496,934,3,0.0,780.67,165.33
4324,CUST5997,798,1,2025-07-20,2025-07-20,0,386,1,0.0,798.00,0.00
4325,CUST5998,1653,4,2024-05-06,2025-06-02,392,434,3,25.0,413.25,98.00


In [4]:
monetary_scalar = MinMaxScaler()
frequency_scalar = MinMaxScaler()
recency_scalar = MinMaxScaler()
diversity_scalar = MinMaxScaler()

total_revenue_normalized = monetary_scalar.fit_transform(customer_metrics[['total_revenue']]).flatten()
total_orders_normalized = frequency_scalar.fit_transform(customer_metrics[['total_orders']]).flatten()
recency_normalized = recency_scalar.fit_transform(customer_metrics[['days_since_last_purchase_recency']]).flatten()
category_diversity_normalized = diversity_scalar.fit_transform(customer_metrics[['category_diversity']]).flatten()

customer_metrics['value_score'] = (
    0.30 * total_revenue_normalized +
    0.25 * total_orders_normalized +
    0.25 * (1 - recency_normalized) +
    0.20 * category_diversity_normalized
).round(3)
customer_metrics

,customer_id,total_revenue,total_orders,first_order_date,last_order_date,days_between_first_and_last_order_date,days_since_last_purchase_recency,category_diversity,return_rate_pct,aov,purchase_frequency,value_score
0,CUST1000,6336,5,2022-12-29,2024-12-30,732,588,3,40.0,1267.20,146.40,0.537
1,CUST1001,690,2,2024-11-08,2025-04-08,151,489,1,0.0,345.00,75.50,0.264
2,CUST1002,999,1,2022-11-09,2022-11-09,0,1370,1,0.0,999.00,0.00,0.038
3,CUST1003,444,2,2023-06-05,2023-12-30,208,954,2,50.0,222.00,104.00,0.193
4,CUST1004,796,1,2025-01-14,2025-01-14,0,573,1,0.0,796.00,0.00,0.216
...,...,...,...,...,...,...,...,...,...,...,...,...
4322,CUST5993,796,1,2023-09-30,2023-09-30,0,1045,1,100.0,796.00,0.00,0.108
4323,CUST5994,2342,3,2022-09-10,2024-01-19,496,934,3,0.0,780.67,165.33,0.309
4324,CUST5997,798,1,2025-07-20,2025-07-20,0,386,1,0.0,798.00,0.00,0.259
4325,CUST5998,1653,4,2024-05-06,2025-06-02,392,434,3,25.0,413.25,98.00,0.440


In [7]:
# step 3
conditions = [
    # Champions: Top 10% by value score
    (customer_metrics['value_score'] >= customer_metrics['value_score'].quantile(0.90)), # Top 10%
    
    # Loyal Customers: High frequency, medium-high value
    (customer_metrics['total_orders'] >= customer_metrics['total_orders'].median()) &
    (customer_metrics['value_score'] >= customer_metrics['value_score'].median()),
    
    # Potential Loyalists: Recent customers with high value
    (customer_metrics['days_since_last_purchase_recency'] <= customer_metrics['days_since_last_purchase_recency'].quantile(0.25)) &
    (customer_metrics['value_score'] >= customer_metrics['value_score'].median()),
    
    # At Risk: High value but low recency
    (customer_metrics['value_score'] >= customer_metrics['value_score'].median()) &
    (customer_metrics['days_since_last_purchase_recency'] > customer_metrics['days_since_last_purchase_recency'].median()),
    
    # Hibernating: Low recency, low frequency, low value
    (customer_metrics['days_since_last_purchase_recency'] > customer_metrics['days_since_last_purchase_recency'].median()) &
    (customer_metrics['total_orders'] < customer_metrics['total_orders'].median()) &
    (customer_metrics['value_score'] < customer_metrics['value_score'].median())
]

choices = [
    'Champions',
    'Loyal Customers',
    'Potential Loyalists',
    'At Risk',
    'Hibernating'
]

customer_metrics['segment'] = np.select(conditions, choices, default='Other')
customer_metrics

,customer_id,total_revenue,total_orders,first_order_date,last_order_date,days_between_first_and_last_order_date,days_since_last_purchase_recency,category_diversity,return_rate_pct,aov,purchase_frequency,value_score,segment
0,CUST1000,6336,5,2022-12-29,2024-12-30,732,588,3,40.0,1267.20,146.40,0.537,Champions
1,CUST1001,690,2,2024-11-08,2025-04-08,151,489,1,0.0,345.00,75.50,0.264,Other
2,CUST1002,999,1,2022-11-09,2022-11-09,0,1370,1,0.0,999.00,0.00,0.038,Hibernating
3,CUST1003,444,2,2023-06-05,2023-12-30,208,954,2,50.0,222.00,104.00,0.193,Other
4,CUST1004,796,1,2025-01-14,2025-01-14,0,573,1,0.0,796.00,0.00,0.216,Other
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4322,CUST5993,796,1,2023-09-30,2023-09-30,0,1045,1,100.0,796.00,0.00,0.108,Hibernating
4323,CUST5994,2342,3,2022-09-10,2024-01-19,496,934,3,0.0,780.67,165.33,0.309,Loyal Customers
4324,CUST5997,798,1,2025-07-20,2025-07-20,0,386,1,0.0,798.00,0.00,0.259,Other
4325,CUST5998,1653,4,2024-05-06,2025-06-02,392,434,3,25.0,413.25,98.00,0.440,Loyal Customers


In [39]:
# step 4
data = data.drop(columns=['segment'], errors='ignore').merge(customer_metrics[['customer_id', 'segment']], how='left', left_on='customer_id', right_on='customer_id', suffixes=('', ''))

segment_stats = data.groupby('segment').agg(
    customer_count=('customer_id', 'nunique'),
    average_rating=('rating', 'mean'),
    total_orders=('order_id', 'count'),
    total_revenue=('revenue', 'sum'),
    avg_order_value=('revenue', 'mean')
).round(2)

def get_most_preferred_category_and_payment_method(group):
    return pd.Series(
        {
            'most_preferred_category': group['category'].mode()[0] if not group['category'].mode().empty else 'N/A',
            'favorite_payment_method': group['payment_method'].mode()[0] if not group['payment_method'].mode().empty else 'N/A'
        }
    )
most_preferred_category_and_payment_method = data.groupby('segment').apply(get_most_preferred_category_and_payment_method, include_groups=False)

order_status_dist = pd.crosstab(data['segment'], data['order_status'], normalize='index') * 100

segment_stats['most_preferred_category'] = most_preferred_category_and_payment_method['most_preferred_category']
segment_stats['favorite_payment_method'] = most_preferred_category_and_payment_method['favorite_payment_method']
for status in order_status_dist.columns:
    segment_stats[f'order_status_{status}_pct'] = order_status_dist[status].round(2)

segment_stats

,customer_count,average_rating,total_orders,total_revenue,avg_order_value,most_preferred_category,favorite_payment_method,order_status_Cancelled_pct,order_status_Delivered_pct,order_status_Pending_pct,order_status_Returned_pct,order_status_Shipped_pct
segment,,,,,,,,,,,,
Champions,434,3.03,2014,1866009,926.52,Electronics,PayPal,19.46,20.85,20.21,19.41,20.06
Hibernating,960,2.95,960,739576,770.39,Electronics,Cash on Delivery,20.83,18.12,20.62,19.79,20.62
Loyal Customers,1716,2.99,4871,3603708,739.83,Electronics,Cash on Delivery,19.81,21.21,19.24,19.38,20.37
Other,1208,2.99,2146,1205109,561.56,Apparel,Cash on Delivery,19.11,18.92,19.85,20.27,21.85
Potential Loyalists,9,1.89,9,36361,4040.11,Electronics,Cash on Delivery,33.33,44.44,0.00,22.22,0.00


### 2. Product Performance & Inventory Optimization

Analyze product performance metrics and recommend inventory optimization strategies.

#### Step 1: Calculate Product Metrics

For each product, calculate the following:

| Metric | Description |
|--------|-------------|
| **Total Units Sold** | Sum of quantity sold across all orders |
| **Total Revenue** | Sum of (quantity × unit_price) |
| **Average Rating** | Mean rating from product reviews |
| **Return Rate** | Returned quantity / Total quantity sold |
| **Cancellation Rate** | Cancelled orders / Total orders |
| **Sales Velocity** | Units sold per month (considering product launch date) |
| **Revenue Concentration** | Percentage of total revenue from this product |


#### Step 2: Product Performance Matrix (2×2 Grid)

Create a **2×2 matrix** with:

- **X-axis:** Average Rating (High/Low based on median)
- **Y-axis:** Revenue Contribution (High/Low based on median)

Categorize products as:

| Quadrant | Category | Description |
|----------|----------|-------------|
| **High-High** | ⭐ **Stars** | High revenue, high rating — winners! |
| **High-Low** | ❓ **Question Marks** | High rating, low revenue — potential |
| **Low-High** | 💰 **Cash Cows** | Low rating, high revenue — need improvement |
| **Low-Low** | 🐕 **Dogs** | Low revenue, low rating — discontinue |


#### Step 3: Inventory Recommendations by Category

For each category, calculate:

- **Inventory Turnover** = `Quantity Sold / Number of Orders Containing Product`

**Recommendations:**

| Category | Recommended Action |
|----------|-------------------|
| **Stars** | ✅ Increase Stock |
| **Question Marks** | 🔍 Maintain (test demand) |
| **Cash Cows** | 📊 Maintain (protect revenue) |
| **Dogs** | ❌ Reduce Stock / Discontinue |

#### Step 4: Identify Underperforming Products

Flag products that meet any of these conditions:

| Condition | Description |
|-----------|-------------|
| **Condition 1** | Rating < 3 AND Return Rate > 20% |
| **Condition 2** | Declining sales trend over last 3 months |
| **Condition 3** | High revenue concentration (>20%) AND declining rating |


#### Step 5: Bundle Recommendation

Find products that are **frequently bought together** using order-level analysis.